<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Miniprojet_week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
Mini-projet : Assistant d’analyse des sentiments avec optimisation de BERT


Bienvenue au mini-projet du dernier jour !

Dans cet atelier, vous perfectionnerez vos compétences bert-base-uncaseden matière de critiques de films, évaluerez le modèle et l'appliquerez à des scénarios concrets de service client. Chaque section explique la raison d'être de chaque étape, les points à observer et propose des illustrations pour faciliter la compréhension du concept par vos apprenants.



Prérequis et mise en place de l'histoire
Scénario : L’équipe d’analyse du support souhaite disposer d’un signal fiable de ressenti pour les commentaires longs afin de pouvoir remonter les problèmes des clients mécontents avant qu’ils ne se désabonnent.
Configuration requise : Python 3.9+, un environnement d’exécution compatible GPU (Colab, Kaggle ou une machine équipée d’un GPU local) et environ 6 Go de VRAM libre. L’utilisation du CPU uniquement est possible, mais l’entraînement sera plus long.
Packages : tensorflow , tensorflow-datasets, transformers, accelerate, et evaluate, qui sont tous apparus plus tôt dans le cours, vous avez donc déjà utilisé ces outils auparavant.


# Run once in a fresh environment
pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate


🔍 Vous remarquerez ici que nous réutilisons exactement la même chaîne d'outils que les jours 3 et 4. Cela renforce la continuité et nous permet de nous concentrer sur le nouveau flux de travail plutôt que sur de nouvelles bibliothèques.

Voici une illustration rapide pour planter le décor de la méthodologie que nous allons explorer :

texte alternatif



Vérification des importations et du matériel
Nous commençons toujours par vérifier les versions et le matériel. Si un apprenant voit cela GPU devices: [], il sait immédiatement qu'il doit changer d'environnement d'exécution (lorsque nous utilisons Google Colab, aucune autre installation ni configuration n'est nécessaire).



import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))


texte alternatif



Charger l'ensemble de données des critiques IMDB
Nous utilisons IMDb car ses avis sont équilibrés (25 000 positifs / 25 000 négatifs) et déjà segmentés. Les apprenants devraient le reconnaître grâce aux exemples d'analyse de sentiments précédents.

(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)


💬 Ici, vous remarquerez que TFDS renvoie à la fois les objets du jeu de données et leurs métadonnées. Notez que cela as_supervised=Trueproduit (text, label)des paires, exactement ce que notre modèle attend.

Jetez un coup d'œil rapide à ces exemples pour concrétiser vos idées :



for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")


Configuration du tokenizer et du pipeline de données
BERT utilise la tokenisation WordPiece pour gérer les mots rares ou inconnus en les décomposant en sous-mots, garantissant ainsi une couverture complète et une taille de vocabulaire optimale. Il ajoute [CLS]des [SEP]tokens pour marquer les limites des phrases et permettre des tâches telles que la classification et la modélisation de paires de phrases. Les masques d'attention indiquent au modèle quels tokens sont valides et lesquels sont des tokens de remplissage, assurant ainsi que l'attention ne soit calculée que sur les entrées valides.



MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)


🧠 Nous importons ce tokenizer pour réutiliser le même vocabulaire que celui appris par le modèle de base en 2018.

Ensuite, convertissez les octets bruts en identifiants de jetons, masques d'attention et identifiants de segments.



def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)


🗒️ N'oubliez pas que cela tf.py_functionnous permet de conserver la logique de tokenisation Hugging Face au sein d'un pipeline TensorFlow, évitant ainsi de manipuler manuellement les tableaux NumPy. De plus, le brassage et le préchargement stabilisent le débit d'entraînement.



Initialiser le modèle de réglage fin
Nous chargeons maintenant TFBertForSequenceClassification, qui regroupe déjà l'encodeur et la tête de classification.



model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


💡 Nous importons ce modèle afin de réutiliser les 110 millions de paramètres appris sur BooksCorpus et Wikipédia. Nous effectuons un réglage fin uniquement sur quelques époques, ce qui explique les taux d'apprentissage de l'ordre de 2e-5.



Former et superviser
Sur un GPU T4 (celui utilisé dans Google Colab), deux époques prennent environ 15 minutes. Veuillez surveiller la précision des calculs d'entraînement et de validation.



EPOCHS = 2  # increase to 3 if time allows
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)


📈 Indiquez comment le plateau de précision de la validation signale le moment d'arrêter. Encouragez également la publication de captures d'écran des courbes d'apprentissage des portefeuilles.



Évaluer sur l'ensemble de test mis de côté
Même si model.fitdes indicateurs de validation sont déjà fournis, nous relançons l'évaluation sur le pipeline de test intact afin de simuler l'assurance qualité en production.



eval_metrics = model.evaluate(test_ds)


✅ *Vous pourrez constater ici si la précision dépasse le seuil de référence de ~0,90 en classe.

#À faire : Utiliser ceci pour discuter des taux d’erreur acceptables pour les équipes de support réelles.*



Créer un assistant d'inférence réutilisable
Intégrez le tout dans une fonction afin de pouvoir coller de véritables transcriptions de justificatifs et obtenir un score instantané.



def predict_sentiment(text: str):
    # Encode the input text
    encoded_input = encode_review(text)

    # Convert to TensorFlow tensors and add batch dimension
    input_ids = tf.constant(encoded_input['input_ids'], dtype=tf.int32)[tf.newaxis, :]
    attention_mask = tf.constant(encoded_input['attention_mask'], dtype=tf.int32)[tf.newaxis, :]
    token_type_ids = tf.constant(encoded_input['token_type_ids'], dtype=tf.int32)[tf.newaxis, :]

    # Create the input dictionary for the model
    model_input = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids
    }

    # Make prediction
    outputs = model.predict(model_input)
    logits = outputs.logits

    # Apply softmax to get probabilities
    probs = tf.nn.softmax(logits, axis=-1)[0]

    # Get the predicted label (0 for negative, 1 for positive)
    predicted_label_id = tf.argmax(probs).numpy()
    label = "Positive" if predicted_label_id == 1 else "Negative"

    return label, float(probs[predicted_label_id].numpy())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")


Pour l'exemple précédent, vous devriez voir Prediction: Positive (confidence=0.5) slightly more or less.

🧭 Les scores de confiance sont essentiels pour décider s'il faut répondre automatiquement ou faire appel à un humain.



Réflexion et prochaines étapes
Pourquoi le réglage fin est important : Vous avez réutilisé un point de contrôle public pour atteindre une précision supérieure à 90 % avec un minimum de données.
Compétences transférables : Tout ce qui précède s’applique également aux tâches de classification dans les domaines des RH, du juridique ou de l’analyse de produits.
Ce que vous pouvez faire avec ceci : adaptation de domaine (collecte des e-mails de votre entreprise), points de contrôle multilingues (DistilBERT multilingue, XLM-R) et surveillance (enregistrement des dérives de données, création de tableaux de bord).
#À faire : Répondre aux questions de réflexion suivantes dans les cellules Markdown de votre cahier :

Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?
Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?
Quels sont les acteurs qui en bénéficient le plus (responsable du support, chef de produit, responsable de la conformité) ?

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 104)

### Réponses aux questions de réflexion :

**Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?**

Le réglage fin d'un modèle pré-entraîné comme BERT, avec un nombre d'époques réduit et des hyperparamètres d'apprentissage spécifiques (taux d'apprentissage faible), a généralement le plus grand impact sur l'amélioration des résultats pour des tâches spécifiques. L'utilisation du tokenizer et du modèle `bert-base-uncased` pré-entraîné sur un vaste corpus de texte permet au modèle de déjà comprendre des concepts linguistiques complexes. Le réglage fin adapte ensuite cette connaissance au domaine spécifique des critiques de films (ou tout autre domaine), ce qui est beaucoup plus efficace que de nettoyer des données ou de chercher des hyperparamètres sur un modèle entraîné de zéro.


**Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?**

Avant de déployer ce signal de sentiment en production, j'ajouterais les garde-fous suivants :

1.  **Seuils de confiance :** Ne considérer les prédictions que si la confiance est supérieure à un certain seuil (par exemple, 0,8 ou 0,9). Pour les prédictions dont la confiance est faible, un humain devrait toujours revoir le commentaire.
2.  **Surveillance continue des performances :** Mettre en place un système pour surveiller la précision et le rappel du modèle sur de nouvelles données entrantes (idéalement annotées manuellement) afin de détecter toute dérive du modèle ou changement dans la distribution des données.
3.  **Gestion des erreurs :** Établir des protocoles pour gérer les cas où le modèle ne parvient pas à faire une prédiction ou produit une sortie inattendue.
4.  **Explicabilité :** Utiliser des techniques d'explicabilité (comme LIME ou SHAP) pour comprendre pourquoi le modèle prend certaines décisions, ce qui peut aider à identifier les biais ou les erreurs.
5.  **Tests A/B :** Déployer le modèle initialement auprès d'un petit sous-ensemble d'utilisateurs ou de cas d'utilisation pour évaluer son impact réel et son acceptation avant un déploiement complet.
6.  **Gestion des langues :** S'assurer que le modèle est capable de gérer différentes langues si le contexte de production l'exige, ou mettre en place des modèles spécifiques par langue.


**Quels sont les acteurs qui en bénéficient le plus (responsable du support, chef de produit, responsable de la conformité) ?**

Ce signal de sentiment peut bénéficier à plusieurs acteurs :

*   **Responsable du support :** Bénéficie le plus directement en identifiant rapidement les clients mécontents ou les problèmes urgents, permettant ainsi une intervention proactive. Cela peut améliorer la satisfaction client et réduire le taux de désabonnement.
*   **Chef de produit :** Peut utiliser ces informations pour identifier les points faibles des produits ou services mentionnés dans les commentaires, prioriser les fonctionnalités ou les corrections, et comprendre les réactions des utilisateurs aux nouvelles versions.
*   **Responsable de la conformité :** Peut utiliser l'analyse de sentiment pour détecter les commentaires qui pourraient signaler des problèmes de conformité, des discours haineux, des menaces ou d'autres contenus inappropriés nécessitant une intervention.
*   **Équipe marketing :** Peut analyser les sentiments pour évaluer la perception des campagnes, des promotions et des marques en général.

### Réponses aux questions de réflexion :

**Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?**

Le réglage fin d'un modèle pré-entraîné comme BERT, avec un nombre d'époques réduit et des hyperparamètres d'apprentissage spécifiques (taux d'apprentissage faible), a généralement le plus grand impact sur l'amélioration des résultats pour des tâches spécifiques. L'utilisation du tokenizer et du modèle `bert-base-uncased` pré-entraîné sur un vaste corpus de texte permet au modèle de déjà comprendre des concepts linguistiques complexes. Le réglage fin adapte ensuite cette connaissance au domaine spécifique des critiques de films (ou tout autre domaine), ce qui est beaucoup plus efficace que de nettoyer des données ou de chercher des hyperparamètres sur un modèle entraîné de zéro.


**Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?**

Avant de déployer ce signal de sentiment en production, j'ajouterais les garde-fous suivants :

1.  **Seuils de confiance :** Ne considérer les prédictions que si la confiance est supérieure à un certain seuil (par exemple, 0,8 ou 0,9). Pour les prédictions dont la confiance est faible, un humain devrait toujours revoir le commentaire.
2.  **Surveillance continue des performances :** Mettre en place un système pour surveiller la précision et le rappel du modèle sur de nouvelles données entrantes (idéalement annotées manuellement) afin de détecter toute dérive du modèle ou changement dans la distribution des données.
3.  **Gestion des erreurs :** Établir des protocoles pour gérer les cas où le modèle ne parvient pas à faire une prédiction ou produit une sortie inattendue.
4.  **Explicabilité :** Utiliser des techniques d'explicabilité (comme LIME ou SHAP) pour comprendre pourquoi le modèle prend certaines décisions, ce qui peut aider à identifier les biais ou les erreurs.
5.  **Tests A/B :** Déployer le modèle initialement auprès d'un petit sous-ensemble d'utilisateurs ou de cas d'utilisation pour évaluer son impact réel et son acceptation avant un déploiement complet.
6.  **Gestion des langues :** S'assurer que le modèle est capable de gérer différentes langues si le contexte de production l'exige, ou mettre en place des modèles spécifiques par langue.


**Quels sont les acteurs qui en bénéficient le plus (responsable du support, chef de produit, responsable de la conformité) ?**

Ce signal de sentiment peut bénéficier à plusieurs acteurs :

*   **Responsable du support :** Bénéficie le plus directement en identifiant rapidement les clients mécontents ou les problèmes urgents, permettant ainsi une intervention proactive. Cela peut améliorer la satisfaction client et réduire le taux de désabonnement.
*   **Chef de produit :** Peut utiliser ces informations pour identifier les points faibles des produits ou services mentionnés dans les commentaires, prioriser les fonctionnalités ou les corrections, et comprendre les réactions des utilisateurs aux nouvelles versions.
*   **Responsable de la conformité :** Peut utiliser l'analyse de sentiment pour détecter les commentaires qui pourraient signaler des problèmes de conformité, des discours haineux, des menaces ou d'autres contenus inappropriés nécessitant une intervention.
*   **Équipe marketing :** Peut analyser les sentiments pour évaluer la perception des campagnes, des promotions et des marques en général.